### 08 — Full System Integration

What this notebook does:

- load all trained model components
- load the selected U-Net segmentation threshold
- prepare the ClinicalBERT and GPT-2 tokenizers
- combine U-Net, EfficientNet, ClinicalBERT, cross-attention, classification, and report generation into one inference pipeline
- automatically segment the lesion from a new ultrasound image
- generate the multimodal feature representation
- predict whether the lesion is benign or malignant
- generate a structured report
- append the classifier result as the final classification
- visualise the ultrasound image, predicted lesion mask, and overlay.


### 1. Imports and Project Setup


In [ ]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
from transformers import AutoTokenizer, GPT2Tokenizer

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

IMAGE_DIR = RAW_DIR / "images"
MODEL_DIR = PROJECT_ROOT / "models"

from src.unet import UNet
from src.image_encoder import EfficientNetEncoder
from src.clinical_encoder import ClinicalBERTEncoder
from src.fusion import CrossAttentionFusion
from src.classifier import MultimodalClassifier
from src.report_generator import MultimodalReportGenerator

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Project root:", PROJECT_ROOT)
print("Images:", IMAGE_DIR.exists())
print("Models:", MODEL_DIR.exists())
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


### 2. Load the Tokenizers

ClinicalBERT tokenises the structured clinical information.

GPT-2 is used for report generation. Its EOS token is used as the padding token, and the custom `<END_REPORT>` token marks the end of the generated report.


In [ ]:
CLINICAL_MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
GPT2_MODEL_NAME = "gpt2"

clinical_tokenizer = AutoTokenizer.from_pretrained(
    CLINICAL_MODEL_NAME
)

gpt2_tokenizer = GPT2Tokenizer.from_pretrained(
    GPT2_MODEL_NAME
)

gpt2_tokenizer.pad_token = (
    gpt2_tokenizer.eos_token
)

gpt2_tokenizer.add_special_tokens({
    "additional_special_tokens": [
        "<END_REPORT>"
    ]
})

END_REPORT_ID = (
    gpt2_tokenizer.convert_tokens_to_ids(
        "<END_REPORT>"
    )
)


### 3. Load the Trained U-Net and Segmentation Threshold

The best U-Net checkpoint is loaded and frozen.

The validation-selected segmentation threshold is loaded from `segmentation_config.json` so that inference uses the same mask threshold as the segmentation stage.


In [ ]:
segmentation_model = UNet().to(device)

segmentation_model.load_state_dict(
    torch.load(
        MODEL_DIR / "best_unet.pth",
        map_location=device
    )
)

segmentation_model.eval()

for parameter in segmentation_model.parameters():
    parameter.requires_grad = False

with open(
    MODEL_DIR / "segmentation_config.json",
    "r"
) as f:
    segmentation_config = json.load(f)

BEST_THRESHOLD = segmentation_config[
    "best_threshold"
]

print("Loaded BEST_THRESHOLD:", BEST_THRESHOLD)


### 4. Load the Multimodal Classifier Components

The image encoder, cross-attention fusion model, and binary classifier are recreated using the same configuration as the classification stage:

- EfficientNet output size: `768`
- fusion embedding size: `768`
- attention heads: `8`
- fusion dropout: `0.2`
- classifier input size: `1536`
- classifier dropout: `0.5`.

Their trained weights are restored from `best_multimodal_classifier.pth`.


In [ ]:
image_encoder = EfficientNetEncoder(
    output_dim=768
).to(device)

fusion_model = CrossAttentionFusion(
    embed_dim=768,
    num_heads=8,
    dropout=0.2
).to(device)

classifier = MultimodalClassifier(
    input_dim=1536,
    dropout=0.5
).to(device)

checkpoint = torch.load(
    MODEL_DIR / "best_multimodal_classifier.pth",
    map_location=device
)

image_encoder.load_state_dict(
    checkpoint["image_encoder"]
)

fusion_model.load_state_dict(
    checkpoint["fusion_model"]
)

classifier.load_state_dict(
    checkpoint["classifier"]
)

image_encoder.eval()
fusion_model.eval()
classifier.eval()


### 5. Load ClinicalBERT

ClinicalBERT is loaded using the same pretrained model used during multimodal training.

In [ ]:
clinical_encoder = ClinicalBERTEncoder(
    model_name=CLINICAL_MODEL_NAME
).to(device)

clinical_encoder.eval()

for parameter in clinical_encoder.parameters():
    parameter.requires_grad = False


### 6. Load the Trained Report Generator

The multimodal report generator is recreated with GPT-2 and a `1536`-dimensional multimodal input.

The token embedding matrix is resized before loading the checkpoint because the tokenizer contains the extra `<END_REPORT>` token.


In [ ]:
report_generator = MultimodalReportGenerator(
    model_name=GPT2_MODEL_NAME,
    multimodal_dim=1536
).to(device)

report_generator.gpt2.resize_token_embeddings(
    len(gpt2_tokenizer)
)

report_generator.load_state_dict(
    torch.load(
        MODEL_DIR / "best_report_generator.pth",
        map_location=device
    )
)

report_generator.gpt2.config.pad_token_id = (
    gpt2_tokenizer.eos_token_id
)

report_generator.eval()

print("Tokenizer size:", len(gpt2_tokenizer))
print(
    "GPT-2 embedding size:",
    report_generator.gpt2.transformer.wte.weight.shape
)


### 7. Define Shared Preprocessing Helpers

These helper functions:

- load and resize the ultrasound image
- build the ClinicalBERT text input
- pool the clinical token features while ignoring padding.


In [ ]:
IMAGE_SIZE = 224

IMAGENET_MEAN = torch.tensor(
    [0.485, 0.456, 0.406],
    device=device
).view(1, 3, 1, 1)

IMAGENET_STD = torch.tensor(
    [0.229, 0.224, 0.225],
    device=device
).view(1, 3, 1, 1)


def load_image(image_path, image_size=IMAGE_SIZE):
    image = Image.open(
        image_path
    ).convert("L")

    image = image.resize(
        (image_size, image_size),
        Image.BILINEAR
    )

    image = np.array(
        image,
        dtype=np.float32
    ) / 255.0

    return torch.tensor(
        image,
        dtype=torch.float32
    ).unsqueeze(0).unsqueeze(0)


def create_clinical_text(
    age,
    tissue_composition,
    signs,
    symptoms
):
    return (
        f"Patient age: {age} years. "
        f"Tissue composition: {tissue_composition}. "
        f"Clinical signs: {signs}. "
        f"Symptoms: {symptoms}."
    )


def masked_mean_pool(
    tokens,
    attention_mask
):
    mask = attention_mask.unsqueeze(-1).float()

    summed = (tokens * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)

    return summed / counts


### 8. Extract the Multimodal Features

This function runs the complete feature-extraction path:

```text
Ultrasound image                        Clinical text
      ↓                                       ↓
U-Net → predicted lesion mask            ClinicalBERT                
      ↓                                       ↓
EfficientNet → visual tokens            clinical tokens
            ↓                                 ↓                     
                Bidirectional cross-attention
                        ↓
                pooled representations
                        ↓
            1536-dimensional feature vector
```


In [ ]:
@torch.no_grad()
def extract_multimodal_features(
    image,
    clinical_text
):
    image = image.to(device)

    # 1. U-Net segmentation
    segmentation_logits = segmentation_model(
        image
    )

    mask_probability = torch.sigmoid(
        segmentation_logits
    )

    predicted_mask = (
        mask_probability >= BEST_THRESHOLD
    ).float()

    # 2. EfficientNet input
    rgb_image = image.repeat(1, 3, 1, 1)
    rgb_image = (
        rgb_image - IMAGENET_MEAN
    ) / IMAGENET_STD

    # 3. Visual features
    visual_tokens = image_encoder(
        rgb_image,
        predicted_mask
    )

    # 4. Clinical features
    clinical_batch = clinical_tokenizer(
        clinical_text,
        padding=True,
        truncation=True,
        max_length=64,
        return_tensors="pt"
    )

    clinical_batch = {
        key: value.to(device)
        for key, value in clinical_batch.items()
    }

    clinical_tokens = clinical_encoder(
        input_ids=clinical_batch["input_ids"],
        attention_mask=clinical_batch["attention_mask"],
        token_type_ids=clinical_batch.get(
            "token_type_ids"
        )
    )

    # 5. Cross-attention fusion
    visual_fused, clinical_fused = fusion_model(
        visual_tokens,
        clinical_tokens,
        clinical_padding_mask=(
            clinical_batch["attention_mask"] == 0
        )
    )

    # 6. Pool both modalities
    visual_vector = visual_fused.mean(
        dim=1
    )

    clinical_vector = masked_mean_pool(
        clinical_fused,
        clinical_batch["attention_mask"]
    )

    multimodal_features = torch.cat(
        [visual_vector, clinical_vector],
        dim=1
    )

    return (
        multimodal_features,
        predicted_mask,
        mask_probability
    )


### 9. Classification Function

The multimodal classifier produces one logit.

The sigmoid function converts it into a malignancy probability. A threshold of `0.5` is used to produce the final benign/malignant class.


In [ ]:
@torch.no_grad()
def classify_case(
    multimodal_features,
    threshold=0.5
):
    logit = classifier(
        multimodal_features
    )

    probability = torch.sigmoid(
        logit
    ).item()

    predicted_class = (
        "malignant"
        if probability >= threshold
        else "benign"
    )

    return predicted_class, probability


### 10. Report-Generation Function

GPT-2 is conditioned on the projected multimodal feature vector.

Generation:

- starts with the GPT-2 EOS token
- uses greedy decoding
- generates at most `100` new tokens
- stops when either `<END_REPORT>` or the normal GPT-2 EOS token is produced.


In [ ]:
@torch.no_grad()
def generate_report(
    multimodal_features,
    max_new_tokens=100
):
    generated_ids = torch.tensor(
        [[gpt2_tokenizer.eos_token_id]],
        device=device
    )

    for _ in range(max_new_tokens):
        token_embeddings = (
            report_generator
            .gpt2
            .transformer
            .wte(generated_ids)
        )

        multimodal_embedding = (
            report_generator
            .multimodal_projection(
                multimodal_features
            )
            .unsqueeze(1)
        )

        inputs_embeds = torch.cat(
            [
                multimodal_embedding,
                token_embeddings
            ],
            dim=1
        )

        outputs = report_generator.gpt2(
            inputs_embeds=inputs_embeds
        )

        next_token = torch.argmax(
            outputs.logits[:, -1, :],
            dim=-1,
            keepdim=True
        )

        token_id = next_token.item()

        if token_id == END_REPORT_ID:
            break

        generated_ids = torch.cat(
            [generated_ids, next_token],
            dim=1
        )

        if token_id == gpt2_tokenizer.eos_token_id:
            break

    return gpt2_tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True
    ).strip()


### 11. Complete End-to-End Inference Function

`run_full_pipeline()` connects all system components.

For one ultrasound image and its clinical information, it returns:

- predicted benign/malignant class
- malignancy probability
- predicted lesion mask
- segmentation probability map
- generated structured report
- formatted ClinicalBERT input text.

The final benign/malignant result comes from the dedicated classifier and is appended to the GPT-2 report.


In [ ]:
def run_full_pipeline(
    image_path,
    age,
    tissue_composition,
    signs,
    symptoms
):
    clinical_text = create_clinical_text(
        age,
        tissue_composition,
        signs,
        symptoms
    )

    image = load_image(
        image_path
    )

    (
        multimodal_features,
        predicted_mask,
        mask_probability
    ) = extract_multimodal_features(
        image,
        clinical_text
    )

    (
        predicted_class,
        malignant_probability
    ) = classify_case(
        multimodal_features
    )

    report = generate_report(
        multimodal_features
    )

    final_report = (
        report.strip()
        + f" Final classification: {predicted_class}."
    )
    return {
        "classification": predicted_class,
        "malignant_probability": malignant_probability,
        "predicted_mask": (
            predicted_mask
            .squeeze()
            .cpu()
            .numpy()
        ),
        "mask_probability": (
            mask_probability
            .squeeze()
            .cpu()
            .numpy()
        ),
        "report": final_report,
        "clinical_text": clinical_text
    }


### 12. Test the Complete Pipeline

One case from the held-out test set is selected to demonstrate the complete system.


In [ ]:
test_df = pd.read_csv(
    PROCESSED_DIR / "test.csv"
)

row = test_df.iloc[16]

result = run_full_pipeline(
    image_path=(
        IMAGE_DIR / row["Image_filename"]
    ),
    age=row["Age"],
    tissue_composition=row[
        "Tissue_composition"
    ],
    signs=row["Signs"],
    symptoms=row["Symptoms"]
)

print("Predicted class:", result["classification"])
print(
    "Malignancy probability:",
    round(
        result["malignant_probability"],
        4
    )
)

print("\nGenerated report:")
print(result["report"])

print("\nActual class:", row["Classification"])


### 13. Visualise the Segmentation Result

The selected ultrasound case is displayed alongside:

- the predicted U-Net lesion mask
- an overlay showing the predicted lesion region on the original ultrasound image.



In [ ]:
original = Image.open(
    IMAGE_DIR / row["Image_filename"]
).convert("L")

original = original.resize(
    (224, 224)
)

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(original, cmap="gray")
plt.title("Ultrasound")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(
    result["predicted_mask"],
    cmap="gray"
)
plt.title("Predicted Mask")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(original, cmap="gray")
plt.imshow(
    result["predicted_mask"],
    alpha=0.35
)
plt.title(
    result["classification"].capitalize()
)
plt.axis("off")

plt.tight_layout()
plt.show()


### 14. Output from this Notebook

The complete inference workflow is now integrated:

```text
Ultrasound image + clinical data
            ↓
        U-Net
            ↓
   Predicted lesion mask
            ↓
EfficientNet + ClinicalBERT
            ↓
Bidirectional cross-attention
            ↓
  Multimodal representation
       ↙             ↘
 Classification      GPT-2
       ↓               ↓
Benign/Malignant   Structured report
       └───────┬───────┘
               ↓
      Final system output
```
